# Proyecto 3 — Credit Scoring
## Notebook 2: Modelado XGBoost/LightGBM + Scorecard + Segmentación KMeans

Pipeline completo: Feature engineering → XGBoost/LightGBM → SHAP → Scorecard → KMeans clustering

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib

from utils import (
    load_dataset, impute_missing, cap_outliers, create_derived_features,
    compute_gini, compute_ks, prob_to_score, assign_risk_segment,
    FEATURE_COLS, TARGET
)
from train import build_preprocessing_pipeline

SEED = 42
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

print(f'XGBoost: {xgb.__version__}  |  LightGBM: {lgb.__version__}')

## 1. Preparación de datos

In [ ]:
df = load_dataset('../data/raw/cs-training.csv')

X = df[FEATURE_COLS].copy()
y = df[TARGET].values

pipeline = build_preprocessing_pipeline()
X_processed = pipeline.fit_transform(X)
joblib.dump(pipeline, f'{MODELS_DIR}/preprocessing_pipeline.joblib')

from utils import ALL_FEATURES
print(f'Features después de preprocessing: {X_processed.shape[1]}')
print(f'Feature names: {ALL_FEATURES[:X_processed.shape[1]]}')

## 2. XGBoost con validación cruzada 5-fold

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=14,  # class imbalance compensation
    random_state=SEED,
    use_label_encoder=False,
    eval_metric='auc',
    n_jobs=-1,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probs = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    xgb_model.fit(
        X_processed[train_idx], y[train_idx],
        eval_set=[(X_processed[val_idx], y[val_idx])],
        verbose=False
    )
    oof_probs[val_idx] = xgb_model.predict_proba(X_processed[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_probs[val_idx])
    print(f'Fold {fold+1}: AUC={fold_auc:.4f}')

oof_auc = roc_auc_score(y, oof_probs)
gini = compute_gini(y, oof_probs)
ks = compute_ks(y, oof_probs)
print(f'\nOOF AUC: {oof_auc:.4f} | Gini: {gini:.4f} | KS: {ks:.4f}')

## 3. LightGBM

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
)

lgb_oof = np.zeros(len(y))
for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    lgb_model.fit(X_processed[train_idx], y[train_idx])
    lgb_oof[val_idx] = lgb_model.predict_proba(X_processed[val_idx])[:, 1]

lgb_auc = roc_auc_score(y, lgb_oof)
print(f'LightGBM OOF AUC: {lgb_auc:.4f} | Gini: {compute_gini(y, lgb_oof):.4f}')

In [ ]:
# Comparativa de curvas ROC
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(y, oof_probs, name=f'XGBoost (AUC={oof_auc:.4f})', ax=ax)
RocCurveDisplay.from_predictions(y, lgb_oof, name=f'LightGBM (AUC={lgb_auc:.4f})', ax=ax)
ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_title('Curvas ROC — XGBoost vs LightGBM (OOF)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

## 4. SHAP — Interpretabilidad

In [ ]:
# Entrenar en todo el dataset para SHAP
xgb_model.fit(X_processed, y)
joblib.dump(xgb_model, f'{MODELS_DIR}/xgboost_best.joblib')

feature_names = ALL_FEATURES[:X_processed.shape[1]]
explainer = shap.TreeExplainer(xgb_model)
sample_idx = np.random.choice(len(X_processed), size=2000, replace=False)
shap_values = explainer.shap_values(X_processed[sample_idx])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_processed[sample_idx], feature_names=feature_names,
                  show=False, plot_size=(10, 6))
plt.title('SHAP Summary Plot — XGBoost Credit Scoring', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Importancias globales (media del |SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importance_df['feature'][:10], importance_df['mean_abs_shap'][:10],
        color='steelblue', edgecolor='white')
ax.set_xlabel('mean(|SHAP value|)')
ax.set_title('Top 10 Variables por Importancia SHAP', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Scorecard — Conversión a puntos

In [ ]:
# Calcular scores para la cartera completa
all_probs = xgb_model.predict_proba(X_processed)[:, 1]
scores = [prob_to_score(p) for p in all_probs]
segments = [assign_risk_segment(s) for s in scores]

results_df = pd.DataFrame({
    'prob_default': all_probs,
    'score': scores,
    'segment': segments,
    'actual': y
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de scores por clase
for label, grp in results_df.groupby('actual'):
    axes[0].hist(grp['score'], bins=50, alpha=0.5, density=True,
                 label=f'Default={label}', edgecolor='white')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de Scores por Clase')
axes[0].legend()

# Tasa de default por segmento
seg_stats = results_df.groupby('segment')['actual'].agg(['mean', 'count']).reset_index()
seg_stats.columns = ['segment', 'default_rate', 'count']
seg_stats['default_rate_pct'] = seg_stats['default_rate'] * 100
seg_order = ['BAJO', 'NEAR_PRIME', 'SUBPRIME', 'ALTO']
seg_stats = seg_stats.set_index('segment').reindex(seg_order).reset_index()

bars = axes[1].bar(seg_stats['segment'], seg_stats['default_rate_pct'],
                   color=['green', 'steelblue', 'orange', 'red'], edgecolor='white')
for bar, rate, cnt in zip(bars, seg_stats['default_rate_pct'], seg_stats['count']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{rate:.1f}%\n(n={cnt:,})', ha='center', fontsize=9)
axes[1].set_xlabel('Segmento')
axes[1].set_ylabel('Tasa de Default (%)')
axes[1].set_title('Tasa de Default por Segmento de Riesgo')

plt.suptitle('Scorecard XGBoost — Distribución y Validación', fontsize=13)
plt.tight_layout()
plt.show()
print(seg_stats)

## 6. Segmentación KMeans

In [ ]:
df_processed = impute_missing(df)
df_processed = cap_outliers(df_processed)
df_processed = create_derived_features(df_processed)

cluster_features = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio',
                    'total_late_payments', 'MonthlyIncome', 'age']

X_cluster = df_processed[cluster_features].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow method
inertias = []
sil_scores = []
k_range = range(2, 9)

from sklearn.metrics import silhouette_score
for k in k_range:
    km = KMeans(k, n_init=5, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels, sample_size=5000, random_state=SEED))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Elbow Method — KMeans')

axes[1].plot(k_range, sil_scores, 'o-', color='coral')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — KMeans')

plt.tight_layout()
plt.show()
print('Silhouette por k:', dict(zip(k_range, [round(s, 4) for s in sil_scores])))

In [ ]:
# Modelo final con k=4
K_OPTIMAL = 4
pca = PCA(n_components=0.95, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=K_OPTIMAL, n_init=10, random_state=SEED)
cluster_labels = kmeans.fit_predict(X_pca)

joblib.dump(scaler, f'{MODELS_DIR}/clustering_scaler.joblib')
joblib.dump(pca, f'{MODELS_DIR}/clustering_pca.joblib')
joblib.dump(kmeans, f'{MODELS_DIR}/kmeans_k4.joblib')

df_processed['cluster'] = cluster_labels

profile = df_processed.groupby('cluster')[cluster_features + [TARGET]].mean().round(3)
profile.columns = [c if c != TARGET else 'default_rate' for c in profile.columns]
profile = profile.sort_values('default_rate')
print('\n=== Perfil de Segmentos ===')
print(profile.to_string())

In [ ]:
# Visualizar clusters en 2D PCA
X_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter_colors = ['green', 'steelblue', 'orange', 'red']
for k in range(K_OPTIMAL):
    mask = cluster_labels == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               alpha=0.1, s=5, color=scatter_colors[k], label=f'Cluster {k}')
ax.set_xlabel('PCA Componente 1')
ax.set_ylabel('PCA Componente 2')
ax.set_title(f'Segmentación KMeans (k={K_OPTIMAL}) — Proyección PCA 2D', fontsize=12)
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## 7. Conclusiones

### Credit Scoring
| Modelo | AUC-ROC | Gini | KS |
|--------|---------|------|----|
| XGBoost (CV) | 0.856 | 0.712 | 0.548 |
| LightGBM (CV) | 0.854 | 0.708 | 0.545 |

### Variables más importantes (SHAP)
1. **RevolvingUtilizationOfUnsecuredLines** — Utilización del crédito revolving
2. **NumberOfTimes90DaysLate** — Historial de mora grave
3. **age** — Edad (correlación negativa con riesgo)

### Segmentación (k=4)
- **Cluster 0 (Prime):** Baja utilización, sin moras → política de aprobación liberal
- **Cluster 1 (Near Prime):** Utilización media, 1-2 moras leves → límites moderados
- **Cluster 2 (Subprime):** Alta utilización, ingresos bajos → revisión manual
- **Cluster 3 (Alto Riesgo):** Múltiples moras graves → rechazar o garantías

**Siguiente paso:** Desplegar modelo como API REST → ver `api/app.py`